# Model Inference and Checking

Loads the best-performing configuration (Logistic Regression on TF-IDF -- retrains in well under a second, no need to persist a model file) and lets you:
1. Classify your own custom text as Real or Fake
2. Inspect the model's actual test-set errors
3. Spot-check predictions against ground truth

Run `src/day1_load_data.py` through `src/day9_logistic_regression.py` at least once before using this notebook, so the required data/feature files exist.

## 1. Setup and Load Artifacts

In [ ]:
import sys, json
sys.path.insert(0, '../src')
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

from day7_consolidate_features import load_feature_set
from day2_clean_tokenize import process_row  # same cleaning used throughout the project

DATA_DIR = '../data'
REPORTS_DIR = '../reports'

## 2. Load the Best Config and Retrain

Reads the winning feature set + hyperparameter from `day9_logreg_results.json` (same pattern used by `day12`-`day15`), so this always reflects your actual best result, not a hardcoded guess.

In [ ]:
with open(f'{REPORTS_DIR}/day9_logreg_results.json') as f:
    logreg_results = json.load(f)
best = max(logreg_results, key=lambda r: r['accuracy'])
feature_set, C = best['feature_set'], best['C']
print(f'Using feature_set={feature_set}, C={C}  (test accuracy was {best["accuracy"]:.4f})')

X_train, X_test, y_train, y_test = load_feature_set(feature_set)
model = LogisticRegression(C=C, max_iter=1000, random_state=42)
model.fit(X_train, y_train)
print('Model trained.')

## 3. Load the Vocabulary and IDF Values

Needed to convert new, raw text into the same TF-IDF feature space the model was trained on.

In [ ]:
with open(f'{DATA_DIR}/bow_vocabulary.json') as f:
    vocabulary = json.load(f)
idf = np.load(f'{DATA_DIR}/idf_values.npy')
print(f'Vocabulary size: {len(vocabulary)}')

## 4. Build a Prediction Function

Reuses the exact same cleaning/tokenization pipeline as the rest of the project (`day2_clean_tokenize.process_row`), so predictions are consistent with how the model was trained.

In [ ]:
from scipy import sparse

def text_to_tfidf_vector(raw_text):
    tokens = process_row(raw_text)
    doc_len = len(tokens)
    vec = np.zeros(len(vocabulary))
    if doc_len == 0:
        return sparse.csr_matrix(vec)
    from collections import Counter
    counts = Counter(tokens)
    for word, count in counts.items():
        col_idx = vocabulary.get(word)
        if col_idx is not None:
            tf = count / doc_len
            vec[col_idx] = tf * idf[col_idx]
    norm = np.linalg.norm(vec)
    if norm > 0:
        vec = vec / norm
    return sparse.csr_matrix(vec)

def predict_fake_or_real(raw_text, verbose=True):
    vec = text_to_tfidf_vector(raw_text)
    prob_fake = model.predict_proba(vec)[0, 1]
    label = 'FAKE' if prob_fake >= 0.5 else 'REAL'
    if verbose:
        print(f'Prediction: {label}   (P(fake) = {prob_fake:.3f})')
    return label, prob_fake

## 5. Try It Yourself

Edit the text below and re-run the cell to test the model on your own examples.

In [ ]:
predict_fake_or_real(
    "WASHINGTON (Reuters) - The Senate voted on Tuesday to advance a bipartisan "
    "infrastructure bill after months of negotiation between lawmakers."
)

predict_fake_or_real(
    "You won't believe what this celebrity said about the government, click to "
    "see the shocking photos that they don't want you to see!"
)

## 6. Spot-Check Against Real Test-Set Articles

Pulls a few random test-set articles and compares the model's prediction to the true label.

In [ ]:
test_df = pd.read_csv(f'{DATA_DIR}/test.csv')
sample = test_df.sample(5, random_state=7)

for _, row in sample.iterrows():
    true_label = 'FAKE' if row['label'] == 1 else 'REAL'
    pred_label, prob = predict_fake_or_real(row['text'], verbose=False)
    correct = '✓' if pred_label == true_label else '✗ MISCLASSIFIED'
    print(f"{correct}  true={true_label:4s}  pred={pred_label:4s}  "
          f"P(fake)={prob:.3f}  title={row['title'][:70]}")

## 7. Full Test-Set Evaluation

Confirms this notebook's retrained model reproduces the metrics reported in the project (should match `day9_logreg_results.json` closely).

In [ ]:
preds = model.predict(X_test)
print(classification_report(y_test, preds, target_names=['Real', 'Fake']))
print('Confusion matrix [[TN, FP], [FN, TP]]:')
print(confusion_matrix(y_test, preds))

## 8. Inspect Misclassified Examples

Useful for spot-checking *why* the model got something wrong -- e.g., does it lack the usual formatting cues discussed in the report's Discussion section?

In [ ]:
preds_full = model.predict(X_test)
test_df_reset = test_df.reset_index(drop=True)
misclassified_idx = np.where(preds_full != y_test)[0]
print(f'{len(misclassified_idx)} / {len(y_test)} test articles misclassified\n')

for idx in misclassified_idx[:5]:
    row = test_df_reset.iloc[idx]
    true_label = 'FAKE' if row['label'] == 1 else 'REAL'
    pred_label = 'FAKE' if preds_full[idx] == 1 else 'REAL'
    print(f"true={true_label}  pred={pred_label}  title={row['title'][:80]}")